# GitAgent — Overview & Example Usage

GitAgent is a git-backed memory management system for agent sessions. It's part of a personal learning project (`PersonalWorkspace`) whose goal is to master LLMs, RAG, and agentic systems by dogfooding self-built tools.

**Two separate layers:**
1. **Build-time** — Claude Code writes and iterates on the GitAgent codebase itself (`gitagent/cli.py`, `gitagent/tools.py`, `gitagent/search.py`, `gitagent/visualize.py`).
2. **Run-time** — GitAgent calls a **local Ollama model** to do its work: `llama3.1:8b` for branch-close summarization, and a dedicated embedding model (`nomic-embed-text`) for semantic search. Claude is never involved at run time.

**Core idea:** use real `git` as the substrate for agent memory, instead of reinventing branching/versioning. A "branch" of work gets a real `git branch` that actually diverges — its `MEMORY.md` only ever lives in that branch's own history. Branches can nest: you can open one from another open branch instead of always from `main`. Only a *top-level* branch (one opened straight off `main`) gets a row in `STATE_TRACKER.md` — nesting can go arbitrarily deep underneath it without `STATE_TRACKER.md` growing a row per branch; the detail lives one hop away, in that top-level branch's own `MEMORY.md`.

This notebook walks through the whole toolset end to end — opening, nesting, updating, closing, abandoning, visualizing, and semantically searching — against a disposable sandbox repo, so it's safe to re-run.

## Directory layout

```text
workspace/
├── STATE_TRACKER.md          # main line — one row per TOP-LEVEL branch only, read/written on `main`
├── pyproject.toml            # packaging; `pip install -e .` gives a real `gitagent` command
├── branch_graph.html         # generated by `gitagent graph` — gitignored, regenerate anytime
├── dashboard.html            # generated by `gitagent dashboard` — gitignored, regenerate anytime
├── .gitagent/chroma/         # derived vector index — gitignored, rebuild with `gitagent reindex`
├── branches/
│   ├── <branch-name>/
│   │   └── MEMORY.md         # scoped working notes - exists only on that branch's own history;
│   │                         # has its own `## Sub-Branches` section indexing every branch nested under it
│   └── archived/
│       └── <branch-name>/
│           └── MEMORY.md     # moved here on close/abandon, on the *base* branch — raw log, never deleted
├── gitagent/
│   ├── cli.py                # argparse CLI — every subcommand below
│   ├── tools.py              # open/update/close/abandon_branch, list_branches
│   ├── search.py             # RAG layer: embeddings + Chroma vector store (optional extra)
│   ├── visualize.py          # render_branch_graph, render_dashboard
│   ├── init.py               # `gitagent init` project bootstrap
│   └── _git.py               # shared git subprocess wrapper + GitAgentError
└── tests/                    # pytest suite, runs against throwaway temp repos
```

Everything gitignored above is **derived**: the source of truth is always the git-tracked `MEMORY.md` / `STATE_TRACKER.md` files. Delete any of it and a single command rebuilds it.

## The tools

| Tool | Git operation | LLM call? | Ends checked out on | What it does |
|---|---|---|---|---|
| `open_branch(name, description, base="main", dry_run=False)` | `git checkout -b name base` | No | `name` | Creates a real branch off `base`. If `base` is `main`, inserts an `Active` row into `STATE_TRACKER.md` on `main`. Otherwise `name` is nested: no `STATE_TRACKER.md` row at all - instead an `Active` line is appended to the *root* branch's (the ancestor that itself was opened off `main`) `## Sub-Branches` section. |
| `update_branch(name, note, dry_run=False)` | commit on `name` | No | `name` | Checks out `name`, appends a timestamped bullet to its own `MEMORY.md` Decisions Log, commits it — the "commit equivalent" of a working note. |
| `close_branch(name, dry_run=False, squash=None)` | `git merge` + `git mv` + commit | **Yes** — local Ollama | `base` | Summarizes `MEMORY.md` via Ollama and merges `name` into its recorded `base` (a **real** merge - `name`'s commits actually diverged). Marks its `STATE_TRACKER.md` row `Completed` if top-level, or flips its line under the root's `## Sub-Branches` if nested. Either way archives (moves, never deletes) the memory file onto `base`. |
| `abandon_branch(name, reason="", dry_run=False)` | `git mv` + commit, **no merge** | No | `base` | For work that didn't pan out. Archives the memory stamped `Abandoned` and records `reason` as the outcome, but never merges. The branch ref and its commits stay put — nothing is deleted, just left unmerged and marked. |
| `list_branches()` | reads `git show main:STATE_TRACKER.md` | No | *(read-only)* | Returns the branch table as `BranchStatus` objects, with each Active branch's own `## Sub-Branches` lines attached. Backs `gitagent status`. |
| `search_branches(query, n_results=5)` | none | **Yes** — Ollama embeddings | *(read-only)* | Semantic search over archived branch memories. Embeds the query with the same model that embedded the documents and returns the closest matches by cosine similarity. |
| `render_branch_graph(output_path=None)` | reads `git log --all` | No | *(read-only)* | Renders every commit/branch as a standalone, self-contained HTML page (lane-colored graph + scrollable log). |
| `render_dashboard(output_path=None)` | reads `git log` / `git show` | No | *(read-only)* | The branch table *and* that same graph on one self-contained page — status pills, outcomes, nested sub-branches. |

**`dry_run` on the three mutating tools** prints exactly what would happen and changes nothing. It exists at the `tools.py` layer, not just the CLI, because the intended caller is a local model driving these through a tool-calling loop with no human confirming each call.

**Squash on close.** By default `close_branch` uses `git merge --squash`, collapsing a branch's per-note `Update branch:` commits into one so they don't bury the main line. Nothing is lost: the note-by-note history stays on the branch ref (never deleted) and in prose in the archived `MEMORY.md`. Set `GITAGENT_SQUASH_ON_CLOSE=0` or pass `--no-squash` for a traditional `--no-ff` merge commit. Note that `--squash` doesn't *record* a merge, so a squashed branch won't appear in `git branch --merged` even though its content is present.

**Why this scales:** without this, every branch ever opened - no matter how deep the nesting - would get its own permanent `STATE_TRACKER.md` row, which turns it into a burden as a project grows. Instead `STATE_TRACKER.md` stays at exactly one row per top-level side branch, and the full nested history is flattened into that branch's own `## Sub-Branches` section (every descendant at any depth, not just direct children) - one line per branch, not one line per note. By the time a top-level branch itself closes, its `Sub-Branches` section already holds every descendant's own summary, so the single `STATE_TRACKER.md` row Ollama generates for it is a genuine roll-up of the whole tree, not just that branch's own top-level notes.

`STATE_TRACKER.md` is only ever read/written while checked out on `main`. Its *working-tree copy* on any other branch is just whatever `main` looked like when that branch forked - to see the current, canonical table from anywhere, use `git show main:STATE_TRACKER.md` rather than trusting the file on disk unless you're actually on `main`.

## Setup: a disposable sandbox repo

The cells below run the real `gitagent.tools` / `gitagent.visualize` functions, but against a throwaway git repo in a temp directory (not this project's own repo), so you can re-run this notebook freely. We do this by importing the modules and pointing their module-level path constants at the sandbox before calling anything.

In [1]:
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

REPO_ROOT = Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from gitagent import tools, visualize

sandbox = Path(tempfile.mkdtemp(prefix="gitagent_demo_"))
print(f"Sandbox repo: {sandbox}")

subprocess.run(["git", "init", "-q", "-b", "main"], cwd=sandbox, check=True)
subprocess.run(["git", "config", "user.email", "demo@example.com"], cwd=sandbox, check=True)
subprocess.run(["git", "config", "user.name", "GitAgent Demo"], cwd=sandbox, check=True)

state_tracker = sandbox / "STATE_TRACKER.md"
state_tracker.write_text(
    "# Demo State Tracker\n\n"
    "## Side Branches (Features & Quests)\n\n"
    "| Branch ID | Feature / Quest Name | Description | Status | Target Outcome |\n"
    "|---|---|---|---|---|\n",
    encoding="utf-8",
)
subprocess.run(["git", "add", "STATE_TRACKER.md"], cwd=sandbox, check=True)
subprocess.run(["git", "commit", "-q", "-m", "Initial commit"], cwd=sandbox, check=True)

# Point gitagent.tools and gitagent.visualize at the sandbox instead of this repo.
tools.WORKSPACE_ROOT = sandbox
tools.BRANCHES_DIR = sandbox / "branches"
tools.ARCHIVED_BRANCHES_DIR = tools.BRANCHES_DIR / "archived"
tools.STATE_TRACKER_PATH = state_tracker
visualize.WORKSPACE_ROOT = sandbox

# gitagent.search is an optional extra (`pip install -e .[search]`), so import
# it defensively - the rest of the notebook runs fine without it.
try:
    from gitagent import search

    search.ARCHIVED_BRANCHES_DIR = tools.ARCHIVED_BRANCHES_DIR
    search.CHROMA_DIR = sandbox / ".gitagent" / "chroma"
    SEARCH_AVAILABLE = True
except ImportError as exc:
    search, SEARCH_AVAILABLE = None, False
    print(f"(search extra not installed, its section will be skipped: {exc})")

print("Ready.")

Sandbox repo: C:\Users\user\AppData\Local\Temp\gitagent_demo_adiwcd3e
Ready.


## 1. `open_branch` — start scoped work (and nest it)

First a normal top-level branch off `main`. Then a *nested* branch opened off that branch instead of `main` - this is the piece that didn't exist before: `open_branch`'s `base` argument. Notice it does **not** add a second `STATE_TRACKER.md` row - it shows up as a line in the parent's own `MEMORY.md` instead.

In [2]:
parent_memory = tools.open_branch(
    "search-endpoint",
    "Add a /search endpoint backed by the new embeddings index.",
)
print(parent_memory)
print(parent_memory.read_text(encoding="utf-8"))

C:\Users\user\AppData\Local\Temp\gitagent_demo_adiwcd3e\branches\search-endpoint\MEMORY.md
# Branch Memory: search-endpoint

## Branched From
main

## Goal
Add a /search endpoint backed by the new embeddings index.

## Status
Active

## Decisions Log

## Sub-Branches

## Open Questions



In [3]:
# Nested: opened FROM search-endpoint, not from main.
child_memory = tools.open_branch(
    "search-endpoint-ranking",
    "Work out the ranking/scoring formula for search results.",
    base="search-endpoint",
)
print(child_memory.read_text(encoding="utf-8"))

# Branch Memory: search-endpoint-ranking

## Branched From
search-endpoint

## Goal
Work out the ranking/scoring formula for search results.

## Status
Active

## Decisions Log

## Sub-Branches

## Open Questions



In [4]:
# STATE_TRACKER.md is only trustworthy read straight from main - see the
# note above. Still exactly ONE row here, even after opening the nested
# ranking branch - it doesn't get one of its own.
canonical_tracker = subprocess.run(
    ["git", "show", "main:STATE_TRACKER.md"], cwd=sandbox, capture_output=True, text=True, check=True
).stdout
print(canonical_tracker)

# Instead, look at search-endpoint's own MEMORY.md - the ranking branch shows
# up there, under Sub-Branches.
print(parent_memory.read_text(encoding="utf-8"))

# Demo State Tracker

## Side Branches (Features & Quests)

| Branch ID | Feature / Quest Name | Description | Status | Target Outcome |
|---|---|---|---|---|
| Branch-001 | search-endpoint | Add a /search endpoint backed by the new embeddings index. | Active | |

# Branch Memory: search-endpoint

## Branched From
main

## Goal
Add a /search endpoint backed by the new embeddings index.

## Status
Active

## Decisions Log

## Sub-Branches
- **search-endpoint-ranking** — Active — Work out the ranking/scoring formula for search results.

## Open Questions



## 2. `update_branch` — log notes as you go

Each call checks out the target branch, appends one timestamped bullet to the Decisions Log, and makes its own commit there — no LLM call involved, it's pure structured logging.

In [5]:
tools.update_branch(
    "search-endpoint",
    "Chose FAISS over a hosted vector DB — no external dependency for a local-first tool.",
)
tools.update_branch(
    "search-endpoint-ranking",
    "Cosine similarity on the raw embedding, no re-ranking pass for v1.",
)
tools.update_branch(
    "search-endpoint-ranking",
    "Added a basic integration test for the ranking function.",
)
print(child_memory.read_text(encoding="utf-8"))

# Branch Memory: search-endpoint-ranking

## Branched From
search-endpoint

## Goal
Work out the ranking/scoring formula for search results.

## Status
Active

## Decisions Log
- [2026-08-31 19:37 UTC] Cosine similarity on the raw embedding, no re-ranking pass for v1.
- [2026-08-31 19:37 UTC] Added a basic integration test for the ranking function.

## Sub-Branches

## Open Questions



## 3. `close_branch` — summarize, merge, archive

This step calls a **local Ollama model** (default `llama3.1:8b` at `http://localhost:11434`, override with the `GITAGENT_OLLAMA_MODEL` / `GITAGENT_OLLAMA_HOST` env vars) to compress the branch's `MEMORY.md` into 2–3 sentences. The prompt explicitly forbids markdown, `|`, and any preamble like "Here is a summary" — small instruct models tend to add one anyway, so double-check the generated text before trusting it verbatim. These cells need `ollama serve` running with that model pulled; they fail gracefully if it isn't reachable.

Closing the **nested** branch first is the interesting part: it merges into `search-endpoint` (its recorded base), *not* `main`, and its Sub-Branches line on `search-endpoint` flips from Active to Completed - `STATE_TRACKER.md` is untouched.

In [6]:
try:
    archived_child = tools.close_branch("search-endpoint-ranking")
    current = subprocess.run(
        ["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=sandbox, capture_output=True, text=True, check=True
    ).stdout.strip()
    print(f"Archived to: {archived_child}")
    print(f"Ended checked out on: {current}  (merged into its base, not main)")
    print("-" * 60)
    print(archived_child.read_text(encoding="utf-8"))
    print("-" * 60)
    print("search-endpoint's Sub-Branches line for it is now Completed:")
    print(parent_memory.read_text(encoding="utf-8"))
except tools.GitAgentError as exc:
    print(f"close_branch failed (is Ollama running? `ollama serve`): {exc}")

Archived to: C:\Users\user\AppData\Local\Temp\gitagent_demo_adiwcd3e\branches\archived\search-endpoint-ranking\MEMORY.md
Ended checked out on: search-endpoint  (merged into its base, not main)
------------------------------------------------------------
# Branch Memory: search-endpoint-ranking

## Branched From
search-endpoint

## Goal
Work out the ranking/scoring formula for search results.

## Status
Completed

## Decisions Log
- [2026-08-31 19:37 UTC] Cosine similarity on the raw embedding, no re-ranking pass for v1.
- [2026-08-31 19:37 UTC] Added a basic integration test for the ranking function.

## Sub-Branches

## Open Questions

------------------------------------------------------------
search-endpoint's Sub-Branches line for it is now Completed:
# Branch Memory: search-endpoint

## Branched From
main

## Goal
Add a /search endpoint backed by the new embeddings index.

## Status
Active

## Decisions Log
- [2026-08-31 19:37 UTC] Chose FAISS over a hosted vector DB — no externa

In [7]:
# Now close the parent. It merges into main, since main is *its* base - this
# is the one that actually gets a STATE_TRACKER.md row, and Ollama sees the
# ranking branch's rolled-up Sub-Branches summary too, not just the parent's
# own notes.
try:
    archived_parent = tools.close_branch("search-endpoint")
    print(f"Archived to: {archived_parent}")
    print("-" * 60)
    print(subprocess.run(["git", "show", "main:STATE_TRACKER.md"], cwd=sandbox, capture_output=True, text=True, check=True).stdout)
except tools.GitAgentError as exc:
    print(f"close_branch failed (is Ollama running? `ollama serve`): {exc}")

Archived to: C:\Users\user\AppData\Local\Temp\gitagent_demo_adiwcd3e\branches\archived\search-endpoint\MEMORY.md
------------------------------------------------------------
# Demo State Tracker

## Side Branches (Features & Quests)

| Branch ID | Feature / Quest Name | Description | Status | Target Outcome |
|---|---|---|---|---|
| Branch-001 | search-endpoint | Add a /search endpoint backed by the new embeddings index. | Completed | A /search endpoint was added with support from the new embeddings index.  The decision was made to use FAISS over a hosted vector DB to keep the project local-first.  The outcome is a functional /search endpoint with a basic ranking function in place. |



## 4. `abandon_branch` — for work that didn't pan out

Not every branch deserves a merge. `abandon_branch` is `close_branch` minus the merge: it archives the memory stamped `Abandoned`, records *why* in the outcome column, and leaves the branch ref and its commits exactly where they are. Nothing is deleted — the work stays reachable on its own branch forever, just unmerged.

**It deliberately makes no LLM call.** What's worth promoting to the main line here isn't a compression of what the branch did — the archived `MEMORY.md` still holds all of that, one hop away — but *why it was dropped*, which is a fact only you have at abandon time and no summarizer could infer from the log. So it takes a `reason` instead. A useful side effect: abandoning works even with Ollama down.

In [8]:
# A branch that turns out to be a dead end.
tools.open_branch("graphql-gateway", "Try a GraphQL gateway in front of the search endpoint.")
tools.update_branch("graphql-gateway", "Spiked the schema; the N+1 resolver problem is worse than REST.")

# --dry-run first: see exactly what it would do, change nothing.
print(tools.abandon_branch("graphql-gateway", reason="preview", dry_run=True))
print()

archived_dead_end = tools.abandon_branch(
    "graphql-gateway",
    reason="GraphQL added an N+1 resolver problem REST didn't have; not worth the complexity.",
)
print(archived_dead_end.read_text(encoding="utf-8"))

# The branch ref still exists, but was never merged into main.
merged = subprocess.run(
    ["git", "branch", "--merged", "main", "--format=%(refname:short)"],
    cwd=sandbox, capture_output=True, text=True, check=True,
).stdout.split()
all_branches = subprocess.run(
    ["git", "branch", "--format=%(refname:short)"],
    cwd=sandbox, capture_output=True, text=True, check=True,
).stdout.split()
print(f"all branches:     {all_branches}")
print(f"merged into main: {merged}")
print("-> 'graphql-gateway' still exists, but its commits never landed on main.")

[dry run] abandon_branch('graphql-gateway')
  - copy branches/graphql-gateway/MEMORY.md to branches/archived/graphql-gateway/MEMORY.md on 'main' (status -> Abandoned)
  - leave 'graphql-gateway' unmerged, its branch ref and commits intact
  - mark 'graphql-gateway' Abandoned in STATE_TRACKER.md



# Branch Memory: graphql-gateway

## Branched From
main

## Goal
Try a GraphQL gateway in front of the search endpoint.

## Status
Abandoned

## Decisions Log
- [2026-08-31 19:37 UTC] Spiked the schema; the N+1 resolver problem is worse than REST.

## Sub-Branches

## Open Questions

all branches:     ['graphql-gateway', 'main', 'search-endpoint', 'search-endpoint-ranking']
merged into main: ['main']
-> 'graphql-gateway' still exists, but its commits never landed on main.


## 5. Visualizing — `render_branch_graph` and `render_dashboard`

Neither makes an LLM call. Both produce **standalone, self-contained HTML files** — open them straight in a browser, no server, no build step. They're point-in-time snapshots, regenerated whenever you want to look, not live views.

- `render_branch_graph()` reads `git log --all` and lays commits into lanes the way `git log --graph` does, then renders SVG. With real nested branches merged in, the graph has more than one lane.
- `render_dashboard()` puts the `STATE_TRACKER.md` branch table — status pills, outcomes, nested sub-branches — *above* that same graph, so you get current state and history on one page. It's the visual counterpart to `gitagent status`.

In [9]:
graph_path = visualize.render_branch_graph()
dashboard_path = visualize.render_dashboard()
print(f"Graph written to:     {graph_path}")
print(f"Dashboard written to: {dashboard_path}")
print("Both are plain, self-contained HTML files — open either in a browser.")

# The same branch table the dashboard renders, as plain data (this is what
# `gitagent status` prints):
for branch in tools.list_branches():
    print(f"\n[{branch.id}] {branch.name} - {branch.status}")
    print(f"    {branch.description}")
    if branch.outcome:
        print(f"    -> {branch.outcome[:100]}...")
    for sub in branch.sub_branches:
        print(f"    {sub[:100]}")

Graph written to:     C:\Users\user\AppData\Local\Temp\gitagent_demo_adiwcd3e\branch_graph.html
Dashboard written to: C:\Users\user\AppData\Local\Temp\gitagent_demo_adiwcd3e\dashboard.html
Both are plain, self-contained HTML files — open either in a browser.



[Branch-001] search-endpoint - Completed
    Add a /search endpoint backed by the new embeddings index.
    -> A /search endpoint was added with support from the new embeddings index.  The decision was made to u...

[Branch-002] graphql-gateway - Abandoned
    Try a GraphQL gateway in front of the search endpoint.
    -> GraphQL added an N+1 resolver problem REST didn't have; not worth the complexity....


## Using the CLI against the real project

Installing the package (`pip install -e .`) registers a real `gitagent` command; without it, use `python -m gitagent.cli` instead. Run from the repo root:

```bash
gitagent init --name "<project>" --goal "<what you're building>"

gitagent open-branch   <name> "<description>" [-b BASE] [--dry-run]
gitagent update-branch <name> "<note>"        [--dry-run]
gitagent close-branch  <name>   [--squash | --no-squash] [--dry-run]
gitagent abandon-branch <name>  [-r "<reason>"] [--dry-run]

gitagent status [--all]          # active branches (or everything)
gitagent search "<query>" [-n N] # semantic search over archived memories
gitagent reindex                 # rebuild the search index from disk

gitagent graph     [-o OUTPUT]   # commit/branch graph -> HTML
gitagent dashboard [-o OUTPUT]   # branch table + graph -> HTML
```

- `-b/--base` defaults to `main`; pass another open branch's name to nest under it.
- `--dry-run` prints what would happen and changes nothing.
- `--no-squash` keeps every note commit and records a `--no-ff` merge instead of the default squash.

### Configuration

| Env var | Default | Purpose |
|---|---|---|
| `GITAGENT_OLLAMA_HOST` | `http://localhost:11434` | Ollama API base URL |
| `GITAGENT_OLLAMA_MODEL` | `llama3.1:8b` | Summarization / field polishing |
| `GITAGENT_OLLAMA_EMBED_MODEL` | `nomic-embed-text` | Embeddings for `search` / `reindex` |
| `GITAGENT_SQUASH_ON_CLOSE` | `1` | Squash a branch's note commits on close; `0` for `--no-ff` |

### Roadmap / open questions (from `CLAUDE.md`)

- Whether `close_branch`'s summarization should stay on the local model or become a hybrid call to Claude if local quality proves insufficient. **Still open** — deliberately deferred, since a run-time Anthropic call would break the build-time/run-time separation above.
- ~~A future `search_branches` tool over archived branch memories via embeddings~~ — **done**, see section 6.
- Final tech stack for orchestration/vector store (tracked as Branch-003 in `STATE_TRACKER.md`). Chroma + Ollama embeddings is now the answer for the vector-store half.

In [10]:
if not SEARCH_AVAILABLE:
    print("Skipped: install the search extra with `pip install -e .[search]`.")
else:
    try:
        # close_branch/abandon_branch index automatically, but reindex_all() is
        # the backfill path - and it embeds every document in one batch, which
        # is a single Ollama round trip rather than one per branch.
        count = search.reindex_all()
        print(f"Indexed {count} archived branch memories.\n")

        # Both queries are deliberately worded to share little or no vocabulary
        # with the branch memories - if this works, it's matching on meaning.
        for query in [
            "how should search results be ordered?",
            "which vector database did we settle on?",
        ]:
            print(f"query: {query!r}")
            for hit in search.search_branches(query, n_results=2):
                print(f"  [{hit.similarity:.3f}] {hit.name} ({hit.status})")
                print(f"          {hit.snippet[:90]}...")
            print()

        # Worth knowing: only Goal + Decisions Log are embedded, so an abandoned
        # branch's *reason* (which lives in STATE_TRACKER.md, not MEMORY.md) is
        # NOT searchable. Queries like "why did we drop X?" won't find it here.
        print("abandoned branch is indexed, but only by its notes - not its reason:")
        for hit in search.search_branches("graphql gateway resolver", n_results=1):
            print(f"  [{hit.similarity:.3f}] {hit.name} ({hit.status})")
    except tools.GitAgentError as exc:
        print(f"Search failed (is Ollama running with nomic-embed-text pulled?): {exc}")

Indexed 3 archived branch memories.

query: 'how should search results be ordered?'


  [0.620] search-endpoint-ranking (Completed)
          Work out the ranking/scoring formula for search results. - [2026-08-31 19:37 UTC] Cosine s...
  [0.537] search-endpoint (Completed)
          Add a /search endpoint backed by the new embeddings index. - [2026-08-31 19:37 UTC] Chose ...

query: 'which vector database did we settle on?'


  [0.654] search-endpoint (Completed)
          Add a /search endpoint backed by the new embeddings index. - [2026-08-31 19:37 UTC] Chose ...
  [0.530] graphql-gateway (Abandoned)
          Try a GraphQL gateway in front of the search endpoint. - [2026-08-31 19:37 UTC] Spiked the...

abandoned branch is indexed, but only by its notes - not its reason:


  [0.776] graphql-gateway (Abandoned)


## The real git history behind it

Every step above was a real commit. Things to look for in the graph:

- **`Log open of ... under ...` / `Log close of ...`** — the Sub-Branches bookkeeping, landing on the *root* branch's own history rather than on `main`.
- **`Squash-merge branch '...'`** — closing squashes by default, so a branch's per-note `Update branch:` commits are folded into one and don't appear on the base branch's line. They're still there on the branch's own ref: try `git log --oneline search-endpoint-ranking` to see them.
- **`graphql-gateway` hanging off on its own** — the abandoned branch. Its commits are reachable and intact, but they never join the main line.

Run the cell with `squash=False` on the closes if you'd rather see traditional `--no-ff` merge commits with every note commit preserved inline.

In [11]:
log = subprocess.run(
    ["git", "log", "--oneline", "--all", "--graph"],
    cwd=sandbox,
    capture_output=True,
    text=True,
    check=True,
)
print(log.stdout)

* 74c28e2 Update branch: graphql-gateway
* c06fa06 Open branch: graphql-gateway
| * ffc4843 Abandon branch: graphql-gateway
| * ef9c130 Archive branch: graphql-gateway
|/  
* 74152b2 Open branch: graphql-gateway
* 7849c59 Close branch: search-endpoint
* 5ecc893 Archive branch: search-endpoint
* 7f6d9fb Squash-merge branch 'search-endpoint'
| * 0637263 Log close of search-endpoint-ranking under search-endpoint
| * 91ec805 Archive branch: search-endpoint-ranking
| * c28db6f Squash-merge branch 'search-endpoint-ranking'
| * d3e6bdd Update branch: search-endpoint
| | * a1ac2be Update branch: search-endpoint-ranking
| | * 931e37b Update branch: search-endpoint-ranking
| | * 38886da Open branch: search-endpoint-ranking
| |/  
| * ad1d9ae Log open of search-endpoint-ranking under search-endpoint
| * 98db80e Open branch: search-endpoint
|/  
* ce6e8e1 Open branch: search-endpoint
* 7566888 Initial commit



## Using the CLI against the real project

The same four operations are exposed as CLI subcommands, run from the repo root:

```bash
python -m gitagent.cli open-branch <name> "<description>" [-b BASE]
python -m gitagent.cli update-branch <name> "<note>"
python -m gitagent.cli close-branch <name>
python -m gitagent.cli graph [-o OUTPUT]
```

`-b/--base` defaults to `main`; pass another open branch's name to nest under it, e.g. `python -m gitagent.cli open-branch search-endpoint-ranking "..." -b search-endpoint`.

## Roadmap / open questions (from `CLAUDE.md`)

- Whether `close_branch`'s summarization should stay on the local model or become a hybrid call to Claude if local quality proves insufficient.
- A future `search_branches` tool over archived branch memories via embeddings (the RAG learning branch).
- Final tech stack for orchestration/vector store (tracked as Branch-003 in `STATE_TRACKER.md`).

In [12]:
shutil.rmtree(sandbox, ignore_errors=True)
print(f"Cleaned up sandbox: {sandbox}")

Cleaned up sandbox: C:\Users\user\AppData\Local\Temp\gitagent_demo_adiwcd3e
